# Content-Based Movie Recommender (TMDB 5000)
**Idea:** describe every movie as one block of text ("tags": plot words + genres + keywords + top cast + director),
turn that text into a numeric vector, and recommend the movies whose vectors are most similar (cosine similarity).

Pipeline: load & merge -> clean -> parse JSON-like columns -> build tags -> stem -> vectorize -> similarity -> recommend

## 1. Load and merge the data

In [1]:
import pandas as pd

movies = pd.read_csv("tmdb_5000_movies.csv")    # general info: title, overview, genres, keywords...
credits = pd.read_csv("tmdb_5000_credits.csv")  # cast and crew for each movie

# Combine both tables into one, matching rows by title.
# (Rows go from 4803 to 4809 because a few titles appear more than once.
#  Merging on the movie id would avoid that.)
movies = movies.merge(credits, on="title")
movies.shape

(4809, 23)

## 2. Keep only the useful columns

In [2]:
# Budget, revenue, popularity etc. don't describe the *content* of a movie, so drop them.
movies = movies[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]]
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


## 3. Clean missing values and duplicates

In [3]:
movies = movies.dropna()                   # remove rows with any missing value (a few movies have no overview)
movies = movies.drop_duplicates()          # remove exact duplicate rows
movies = movies.reset_index(drop=True)     # renumber rows 0..n-1 so positions match the similarity matrix later
len(movies)

4806

## 4. Parse the JSON-like columns
`genres`, `keywords`, `cast` and `crew` are stored as **strings** that look like a list of dicts,
e.g. `'[{"id": 28, "name": "Action"}, ...]'`. We convert them into plain Python lists of names.

In [4]:
import ast  # ast.literal_eval safely turns a string like "[{...}]" into a real Python list

def names(text):
    """Return just the 'name' of every dict in the string (e.g. all genre names)."""
    return [d["name"] for d in ast.literal_eval(text)]

def director(text):
    """The crew list has many jobs (editor, producer...). Keep only the Director's name."""
    return [d["name"] for d in ast.literal_eval(text)
            if d["job"] == "Director"]

movies["genres"] = movies["genres"].apply(names)
movies["keywords"] = movies["keywords"].apply(names)
movies["cast"] = movies["cast"].apply(lambda x: names(x)[:3])  # only the top 3 billed actors
movies["crew"] = movies["crew"].apply(director)                # crew now only holds the director
movies.iloc[0]  # inspect the first movie (Avatar) to check the result

movie_id                                                19995
title                                                  Avatar
overview    In the 22nd century, a paraplegic Marine is di...
genres          [Action, Adventure, Fantasy, Science Fiction]
keywords    [culture clash, future, space war, space colon...
cast         [Sam Worthington, Zoe Saldana, Sigourney Weaver]
crew                                          [James Cameron]
Name: 0, dtype: object

## 5. Build the "tags" column
All features are merged into one list of words per movie.

In [5]:
# Split the overview sentence into a list of words so it can be added to the other lists.
movies["overview"] = movies["overview"].apply(lambda x: x.split())

# Remove spaces inside names so each name becomes ONE token:
# "Sam Worthington" -> "SamWorthington" (otherwise "Sam" in one movie would match "Sam" in another person's name),
# "Science Fiction" -> "ScienceFiction".
for col in ["genres", "keywords", "cast", "crew"]:
    movies[col] = movies[col].apply(
        lambda x: [i.replace(" ", "") for i in x]
    )

# Concatenate all the lists into one big list of words per movie.
movies["tags"] = (movies["overview"]
    + movies["genres"] + movies["keywords"]
    + movies["cast"] + movies["crew"])

## 6. Lowercase and stem
Stemming reduces words to their root so that "loving", "loved" and "love" count as the same word.

In [6]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

# List of words -> one lowercase string
movies["tags"] = movies["tags"].apply(lambda x: " ".join(x).lower())

# Stem every word in the string
movies["tags"] = movies["tags"].apply(
    lambda x: " ".join(ps.stem(w) for w in x.split()))

# Quick sanity check of the stemmer: both should print "love"
print(ps.stem("loving"), ps.stem("loved"))

love love


## 7. Turn text into vectors (Bag of Words)
Each movie becomes a vector of word counts over the 5000 most frequent words.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000,       # keep only the 5000 most common words
                     stop_words="english")    # ignore filler words like "the", "and", "is"
vectors = cv.fit_transform(movies["tags"]).toarray()
vectors.shape   # (number of movies, 5000)

(4806, 5000)

## 8. Compute similarity between all movies
Cosine similarity measures the angle between two vectors: 1 = identical direction (very similar), 0 = nothing in common.
The result is a square table where `similarity[i][j]` is how alike movie *i* and movie *j* are.

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)
similarity.shape   # (n_movies, n_movies)

(4806, 4806)

## 9. Recommendation function

In [9]:
def recommend(movie):
    """Print the 5 movies most similar to the given title."""
    # Row position of the chosen movie (raises IndexError if the title doesn't exist)
    index = movies[movies["title"] == movie].index[0]

    # Pair each movie's position with its similarity score to the chosen movie
    scores = list(enumerate(similarity[index]))

    # Sort from most to least similar
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # scores[0] is the movie itself (similarity 1.0), so skip it and take the next 5
    for i, score in scores[1:6]:
        print(movies.iloc[i].title)

## 10. Try it

In [10]:
recommend("Falcon Rising")

Highlander
Jarhead
Battle: Los Angeles
Something's Gotta Give
Avatar
